# Module 4 — Knowledge Assistant (Agent Bricks)

**Time**: ~20 minutes

**For analysts**: A Knowledge Assistant is a no-code RAG (retrieval-augmented generation) agent. You point it at a folder of documents — CISA advisories, STIG XML, ATT&CK descriptions — and it answers questions with citations.

**For engineers**: Behind the scenes, Agent Bricks chunks the documents, embeds them, builds a Vector Search index, wires up a retrieval-augmented chat endpoint, and deploys it. Everything you'd build by hand with a vector store + LangChain — managed for you.

**Why we picked KA over a knowledge graph**: a knowledge graph is the right answer for relationship-heavy queries (*"who is connected to whom"*). For DISA's primary need — *"what does this STIG control require"*, *"what does CISA recommend for this CVE"* — a vector-based KA over authoritative documents is faster to set up and easier to maintain.

## 1. Stage the corpus

We combine three document types in one volume: CISA advisories (PDF), STIG XCCDF excerpts (XML), and the MITRE ATT&CK technique descriptions (which we already loaded as a Delta table — we'll re-emit them as small text files for KA).

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS disa_workshop.threat_intel.ka_corpus;
LIST '/Volumes/disa_workshop/threat_intel/raw_advisories'

In [ ]:
import shutil
import os

src_advisories = "/Volumes/disa_workshop/threat_intel/raw_advisories"
src_stigs = "/Volumes/disa_workshop/threat_intel/raw_stigs"
dst = "/Volumes/disa_workshop/threat_intel/ka_corpus"

for src in (src_advisories, src_stigs):
    if not os.path.exists(src):
        continue
    for fn in os.listdir(src):
        shutil.copy(f"{src}/{fn}", f"{dst}/{fn}")
        print(f"Copied {fn}")

# Emit ATT&CK techniques as text snippets
techniques = spark.table("disa_workshop.threat_intel.attack_techniques").collect()
for row in techniques[:50]:  # first 50 to keep corpus small
    fname = f"{dst}/attack_{row.technique_id}.txt"
    with open(fname, "w") as f:
        f.write(f"# {row.technique_id}: {row.name}\n\n{row.description}\n\nPlatforms: {', '.join(row.platforms or [])}")
print(f"Wrote {min(50, len(techniques))} ATT&CK technique files to {dst}")

## 2. Create the Knowledge Assistant

From the workspace UI: **Agents > Knowledge Assistant > Create**.

Configuration:
- **Name**: `disa-cti-knowledge`
- **Source**: `/Volumes/disa_workshop/threat_intel/ka_corpus`
- **Embedding model**: `databricks-gte-large-en` (default)
- **Chunk size**: 1000 tokens (default)
- **Foundation model**: Llama 4 Maverick or Claude Sonnet 4.6
- **System prompt**: paste the prompt below

Wait ~3 minutes for index build.

In [ ]:
KA_SYSTEM_PROMPT = """
You are a DISA cyber threat intelligence assistant. Answer questions about CISA advisories,
DoD STIGs, and MITRE ATT&CK techniques. Always cite the source document name and section.

If the answer is not in the retrieved context, say so explicitly. Do not make up CVE IDs,
STIG identifiers, or ATT&CK technique IDs.

When citing a STIG control, include both the STIG-ID (e.g., V-220701) and the severity (CAT I/II/III).
When citing an ATT&CK technique, include the technique ID (e.g., T1059.001).
""".strip()

print(KA_SYSTEM_PROMPT)
print("\n--- Paste this into the KA system prompt field. ---")

## 3. Test the assistant

Once the KA build completes, save its endpoint name (something like `agents_disa-cti-knowledge`) and query it from this notebook to verify.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

KA_ENDPOINT = "agents_disa-cti-knowledge"  # update with the endpoint name from the KA build screen

TEST_QUESTIONS = [
    "What does CISA recommend for protecting against ransomware on legacy Windows systems?",
    "Which STIG controls cover privileged account auditing on Windows Server?",
    "Summarize the most recent advisory in the corpus and list its CVEs.",
    "What ATT&CK technique is associated with PowerShell-based execution? Include the technique ID.",
    "Are there any advisories that specifically call out Cisco IOS XE vulnerabilities?",
]

for q in TEST_QUESTIONS:
    response = w.serving_endpoints.query(
        name=KA_ENDPOINT,
        messages=[{"role": "user", "content": q}],
    )
    print(f"Q: {q}")
    print(f"A: {response.choices[0].message.content[:500]}...\n{'-' * 60}")

## 4. Why this is different from `ai_query`

Module 1's `ai_query` was a one-shot LLM call against the parsed PDF text we passed in. The KA endpoint is a full retrieval pipeline:

1. Embed the question
2. Retrieve top-K chunks from the Vector Search index
3. Stuff them into a prompt
4. Generate the answer with citations

This means the KA can answer questions that span multiple documents — *"compare CISA's guidance with the relevant STIG"* — which `ai_query` cannot.

**Next**: **Module 5 — `notebooks/05_compound_agent.ipynb`** wraps Genie + KA + a public MCP server into a single agent.